# Data library
#### This file contains the average PGA Tour data, parameter variation library, and TI - z0 conversions


In [1]:
2+2

4

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from Tracer import WindField, Trajectory
from Tracer.solvers import solver_rk45, solver_euler

Synthesizing wind field with parameters:  z_height=100, direction=0, U_ref=10.0, z0=0.03, z_ref=10.0
Synthesizing wind field with parameters:  z_height=100, direction=0, U_ref=10.0, z0=0.03, z_ref=10.0


### PGA-LPGA STATS

In [3]:
# Data from tour averages for the men's (PGA) and women's (LPGA) golf turnaments
# from https://www.trackman.com/blog/golf/introducing-updated-tour-averages

pga_data = [
    ["Driver", 115, -0.9, 171, 1.49, 10.4, 2545, 32, 39, 258],
    ["3-wood", 110, -2.3, 162, 1.47, 9.3, 3663, 29, 44, 228],
    ["5-wood", 106, -2.5, 156, 1.47, 9.7, 4322, 30, 48, 216],
    ["Hybrid", 102, -2.4, 149, 1.47, 10.2, 4587, 28, 49, 211],
    ["3 Iron", 100, -2.5, 145, 1.46, 10.3, 4404, 27, 48, 199],
    ["4 Iron", 98, -2.9, 140, 1.44, 10.8, 4782, 28, 49, 192],
    ["5 Iron", 96, -3.4, 135, 1.41, 11.9, 5280, 30, 50, 182],
    ["6 Iron", 94, -3.7, 130, 1.39, 14.0, 6204, 29, 50, 172],
    ["7 Iron", 92, -3.9, 123, 1.34, 16.1, 7124, 31, 51, 161],
    ["8 Iron", 89, -4.2, 118, 1.33, 17.8, 8078, 30, 51, 150],
    ["9 Iron", 87, -4.3, 112, 1.29, 20.0, 8793, 29, 52, 139],
    ["PW", 84, -4.7, 104, 1.24, 23.7, 9316, 29, 52, 130],
]

lpga_data = [
    ["Driver", 96, 2.8, 143, 1.49, 12.6, 2506, 24, 36, 204],
    ["3-wood", 92, -0.8, 135, 1.47, 11.6, 2595, 23, 38, 183],
    ["5-wood", 90, -1.6, 130, 1.46, 12.3, 4320, 23, 43, 173],
    ["Hybrid", 87, -1.9, 125, 1.44, 13.9, 4504, 23, 45, 163],
    ["4 Iron", 82, -1.7, 118, 1.43, 13.9, 4608, 23, 43, 160],
    ["5 Iron", 81, -2.0, 114, 1.42, 14.6, 4966, 23, 45, 152],
    ["6 Iron", 80, -2.3, 111, 1.41, 16.7, 5904, 23, 46, 142],
    ["7 Iron", 78, -2.5, 106, 1.38, 18.5, 6630, 24, 47, 131],
    ["8 Iron", 76, -2.8, 102, 1.36, 20.8, 7413, 25, 47, 122],
    ["9 Iron", 74, -3.2, 95, 1.30, 23.5, 7605, 25, 48, 112],
    ["PW", 72, -3.2, 88, 1.25, 25.2, 8465, 25, 48, 101],
]

columns = [
    "Club",
    "Club Speed (mph)",
    "Attack Angle (deg)",
    "Ball Speed (mph)",
    "Smash Factor",
    "Launch Angle (deg)",
    "Spin Rate (rpm)",
    "Max Height (m)",
    "Land Angle (deg)",
    "Carry (m)"
]

df_pga = pd.DataFrame(pga_data, columns=columns)
df_lpga = pd.DataFrame(lpga_data, columns=columns)


# convert from mph to m/s
df_pga['Ball Speed (mph)'] = df_pga['Ball Speed (mph)'] * 0.44704
df_pga = df_pga.rename(columns={'Ball Speed (mph)':'Ball Speed (m/s)'})
df_pga['Club Speed (mph)'] = df_pga['Club Speed (mph)'] * 0.44704
df_pga = df_pga.rename(columns={'Club Speed (mph)':'Club Speed (m/s)'})

df_lpga['Ball Speed (mph)'] = df_lpga['Ball Speed (mph)'] * 0.44704
df_lpga = df_lpga.rename(columns={'Ball Speed (mph)':'Ball Speed (m/s)'})
df_lpga['Club Speed (mph)'] = df_lpga['Club Speed (mph)'] * 0.44704
df_lpga = df_lpga.rename(columns={'Club Speed (mph)':'Club Speed (m/s)'})

display(df_pga)
display(df_lpga)


,Club,Club Speed (m/s),Attack Angle (deg),Ball Speed (m/s),Smash Factor,Launch Angle (deg),Spin Rate (rpm),Max Height (m),Land Angle (deg),Carry (m)
0,Driver,51.40960,-0.9,76.44384,1.49,10.4,2545,32,39,258
1,3-wood,49.17440,-2.3,72.42048,1.47,9.3,3663,29,44,228
2,5-wood,47.38624,-2.5,69.73824,1.47,9.7,4322,30,48,216
3,Hybrid,45.59808,-2.4,66.60896,1.47,10.2,4587,28,49,211
4,3 Iron,44.70400,-2.5,64.82080,1.46,10.3,4404,27,48,199
5,4 Iron,43.80992,-2.9,62.58560,1.44,10.8,4782,28,49,192
6,5 Iron,42.91584,-3.4,60.35040,1.41,11.9,5280,30,50,182
7,6 Iron,42.02176,-3.7,58.11520,1.39,14.0,6204,29,50,172
8,7 Iron,41.12768,-3.9,54.98592,1.34,16.1,7124,31,51,161
9,8 Iron,39.78656,-4.2,52.75072,1.33,17.8,8078,30,51,150


,Club,Club Speed (m/s),Attack Angle (deg),Ball Speed (m/s),Smash Factor,Launch Angle (deg),Spin Rate (rpm),Max Height (m),Land Angle (deg),Carry (m)
0,Driver,42.91584,2.8,63.92672,1.49,12.6,2506,24,36,204
1,3-wood,41.12768,-0.8,60.35040,1.47,11.6,2595,23,38,183
2,5-wood,40.23360,-1.6,58.11520,1.46,12.3,4320,23,43,173
3,Hybrid,38.89248,-1.9,55.88000,1.44,13.9,4504,23,45,163
4,4 Iron,36.65728,-1.7,52.75072,1.43,13.9,4608,23,43,160
5,5 Iron,36.21024,-2.0,50.96256,1.42,14.6,4966,23,45,152
6,6 Iron,35.76320,-2.3,49.62144,1.41,16.7,5904,23,46,142
7,7 Iron,34.86912,-2.5,47.38624,1.38,18.5,6630,24,47,131
8,8 Iron,33.97504,-2.8,45.59808,1.36,20.8,7413,25,47,122
9,9 Iron,33.08096,-3.2,42.46880,1.30,23.5,7605,25,48,112


In [4]:
df_lpga[['Ball Speed (m/s)', 'Launch Angle (deg)', 'Spin Rate (rpm)', 'Max Height (m)', 'Carry (m)']].to_numpy()

array([[  63.92672,   12.6    , 2506.     ,   24.     ,  204.     ],
       [  60.3504 ,   11.6    , 2595.     ,   23.     ,  183.     ],
       [  58.1152 ,   12.3    , 4320.     ,   23.     ,  173.     ],
       [  55.88   ,   13.9    , 4504.     ,   23.     ,  163.     ],
       [  52.75072,   13.9    , 4608.     ,   23.     ,  160.     ],
       [  50.96256,   14.6    , 4966.     ,   23.     ,  152.     ],
       [  49.62144,   16.7    , 5904.     ,   23.     ,  142.     ],
       [  47.38624,   18.5    , 6630.     ,   24.     ,  131.     ],
       [  45.59808,   20.8    , 7413.     ,   25.     ,  122.     ],
       [  42.4688 ,   23.5    , 7605.     ,   25.     ,  112.     ],
       [  39.33952,   25.2    , 8465.     ,   25.     ,  101.     ]])

### PARAM VAR

In [72]:
# play field parameters - dont touch unless necessary
P0=np.array([0, 0, 0])
nx = 500        # length of play field
ny = 200        # width of play field
nz = 100        # height of play field
dt = 0.01       # time step

# Basis shot parameters - if not otherwise specified, these are used
shot_speed = 76.44384   # intital velocity
shot_angle = 10.4       # initial angle for trajectory	
shot_spin = 2545        # initial spin

spin_axis = 0

design_n = 10
def include_baseline(values, baseline, design_n=design_n):
    """
    Ensures baseline is included while keeping exactly n_total samples.
    Replaces the nearest sampled value if baseline is not already present.
    """
    values = np.array(values)

    # if baseline already exists, just return sorted values
    if np.isclose(values, baseline).any():
        return np.sort(values)

    # find nearest existing point and replace it
    idx = np.argmin(np.abs(values - baseline))
    values[idx] = baseline

    return np.sort(values)

# Basis wind parameters - if not otherwise specified, these are used
U_ref = 6      # reference wind speed at 10 meters above ground (6 m/s is approx. danish average)
z0 = 0.03       # surface roughness
direction = 0   # wind direction (0 is tailwind, 180 is headwind)

baseline_values = {
    "shot_speed": shot_speed,
    "shot_angle": shot_angle,
    "shot_spin": shot_spin,
    "U_ref": U_ref,
    "z0": z0,
    "direction": direction
}

parameter_library = {
    "shot_speed": {
        "values": include_baseline(np.linspace(35, 85, design_n), shot_speed),
        "name": "Shot speed",
        "unit": "m/s"
    },
    "shot_angle": {
        "values": include_baseline(np.linspace(5, 25, design_n), shot_angle),
        "name": "Shot angle",
        "unit": "deg"
    },
    "shot_spin": {
        "values": include_baseline(np.linspace(2000, 10000, design_n), shot_spin),
        "name": "Shot spin",
        "unit": "rpm"
    },
    "U_ref": {
        "values": include_baseline(np.linspace(2, 10, design_n), U_ref),
        "name": "Wind speed",
        "unit": "m/s"
    },
    "z0": {
        "values": include_baseline(np.logspace(np.log10(0.001), np.log10(1), design_n), z0),
        "name": "Roughness",
        "unit": "m"
    },
    "direction": {
        "values": include_baseline(np.linspace(0, 180, design_n), direction),
        "name": "Wind direction",
        "unit": "deg"
    }
}

summary_rows = []

for key, info in parameter_library.items():
    vals = np.array(info["values"])

    summary_rows.append({
        "Key": key,
        "Parameter": info["name"],
        "Baseline": baseline_values[key],
        "Unit": info["unit"],
        "Min": np.min(vals),
        "Max": np.max(vals),
        "Samples": len(vals)
    })

parameter_summary = pd.DataFrame(summary_rows)
display(parameter_summary)

expanded_rows = []

for key, info in parameter_library.items():
    vals = list(np.round(info["values"], 4))

    row = {
        "Key": key,
        "Parameter": info["name"],
        "Baseline": baseline_values[key],
        "Unit": info["unit"]
    }

    for i, v in enumerate(vals, 1):
        row[f"Value {i}"] = v

    expanded_rows.append(row)

parameter_expanded = pd.DataFrame(expanded_rows)
display(parameter_expanded)

,Key,Parameter,Baseline,Unit,Min,Max,Samples
0,shot_speed,Shot speed,76.44384,m/s,35.000,85.0,10
1,shot_angle,Shot angle,10.40000,deg,5.000,25.0,10
2,shot_spin,Shot spin,2545.00000,rpm,2000.000,10000.0,10
3,U_ref,Wind speed,6.00000,m/s,2.000,10.0,10
4,z0,Roughness,0.03000,m,0.001,1.0,10
5,direction,Wind direction,0.00000,deg,0.000,180.0,10


,Key,Parameter,Baseline,Unit,Value 1,Value 2,Value 3,Value 4,Value 5,Value 6,Value 7,Value 8,Value 9,Value 10
0,shot_speed,Shot speed,76.44384,m/s,35.000,40.5556,46.1111,51.6667,57.2222,62.7778,68.3333,76.4438,79.4444,85.0
1,shot_angle,Shot angle,10.40000,deg,5.000,7.2222,10.4000,11.6667,13.8889,16.1111,18.3333,20.5556,22.7778,25.0
2,shot_spin,Shot spin,2545.00000,rpm,2000.000,2545.0000,3777.7778,4666.6667,5555.5556,6444.4444,7333.3333,8222.2222,9111.1111,10000.0
3,U_ref,Wind speed,6.00000,m/s,2.000,2.8889,3.7778,4.6667,6.0000,6.4444,7.3333,8.2222,9.1111,10.0
4,z0,Roughness,0.03000,m,0.001,0.0022,0.0046,0.0100,0.0300,0.0464,0.1000,0.2154,0.4642,1.0
5,direction,Wind direction,0.00000,deg,0.000,20.0000,40.0000,60.0000,80.0000,100.0000,120.0000,140.0000,160.0000,180.0


### Convertion between TI and z0

In [81]:
kappa = 0.4
cmu= 0.03

def convertTItoz0(TI,zref):
    z0 = (zref)/(-1.0+np.exp((kappa*np.sqrt(6.0))/(3.0*TI*cmu**(1.0/4.0))))
    return z0

def convertZ0toTI(z0,zref):
    TI = (kappa*np.sqrt(2.0/3.0))/((cmu**(1.0/4.0))*np.log((zref+z0)/(z0)))
    return TI

TI_vals = []
for i in range(10):
    TI_vals.append(round(convertZ0toTI(parameter_library["z0"]["values"][i],zref=10),5))

In [ ]:
print('z0 values: ',parameter_library['z0']['values'])
print('TI values: ',TI_vals)

z0 values:  [0.001      0.00215443 0.00464159 0.01       0.03       0.04641589
 0.1        0.21544347 0.46415888 1.        ]
TI values:  [0.0852, 0.09295, 0.10224, 0.11359, 0.13502, 0.14594, 0.17004, 0.20336, 0.25189, 0.32727]
